# A Comparative Study of Methods for Early Breast Cancer Detection Using Medical Data

**Models:** Decision Tree · Random Forest · SVM  
**Dataset:** Wisconsin Breast Cancer Dataset (WBCD)  
**Optimized pipeline:** Acquisition → Preprocessing → Correlation filter → Nested CV tuning (Pipeline) → Hold-out test → Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, make_scorer
)

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
CORR_THRESHOLD = 0.92
F1_MALIGNANT = make_scorer(f1_score, pos_label=1)
sns.set_theme(style="whitegrid")

## 1. Data Acquisition

In [ ]:
df = pd.read_csv("data.csv")
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
empty_cols = [c for c in df.columns if str(c).strip() == ""]
df = df.drop(columns=empty_cols, errors="ignore")
if "id" in df.columns:
    df = df.drop(columns=["id"])

print("Shape:", df.shape)
print(df["diagnosis"].value_counts())
df.head()

## 2. Data Preprocessing

- Encode labels: **Malignant = 1** (positive class), **Benign = 0**
- Stratified 80/20 train–test split
- Scaling + ANOVA feature selection happen **inside** each CV fold via `Pipeline` (no leakage)

In [ ]:
y = df["diagnosis"].map({"M": 1, "B": 0}).astype(int)
X = df.drop(columns=["diagnosis"])

print("Missing values:", int(X.isna().sum().sum()))
print("Benign:", int((y == 0).sum()), "| Malignant:", int((y == 1).sum()))

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.6})
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 3. Feature Selection

1. Drop highly correlated features using **training-set** correlations only  
2. Tune ANOVA `SelectKBest` (k) inside GridSearchCV for each model

In [ ]:
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
feature_names = [c for c in X_train.columns if c not in to_drop]
X_train = X_train[feature_names].copy()
X_test = X_test[feature_names].copy()

print(f"Correlation filter (>{CORR_THRESHOLD}): dropped {len(to_drop)} → {len(feature_names)} features")
if to_drop:
    print("Dropped:", to_drop)

## 4–5. Nested CV Tuning & Hold-out Testing

Each model uses `Pipeline(StandardScaler → SelectKBest → Classifier)` with stratified 5-fold `GridSearchCV`, scored on **malignant-class F1**.

In [ ]:
n_features = X_train.shape[1]
k_options = sorted({min(k, n_features) for k in (10, 15, 20, n_features)})
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

search_spaces = {
    "Decision Tree": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif)),
            ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]),
        {
            "select__k": k_options,
            "clf__max_depth": [3, 5, 8, None],
            "clf__min_samples_split": [2, 5, 10],
            "clf__class_weight": [None, "balanced"],
        },
    ),
    "Random Forest": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif)),
            ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        {
            "select__k": k_options,
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [5, 10, None],
            "clf__min_samples_leaf": [1, 2],
            "clf__max_features": ["sqrt", 0.5],
            "clf__class_weight": [None, "balanced"],
        },
    ),
    "SVM": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif)),
            ("clf", SVC(kernel="rbf", random_state=RANDOM_STATE)),
        ]),
        {
            "select__k": k_options,
            "clf__C": [0.1, 1.0, 10.0],
            "clf__gamma": ["scale", 0.01, 0.1],
            "clf__class_weight": [None, "balanced"],
        },
    ),
}

rows, predictions, fitted = [], {}, {}

for name, (pipe, grid) in search_spaces.items():
    print(f"\n=== Tuning {name} ===")
    search = GridSearchCV(
        pipe, grid, scoring=F1_MALIGNANT, cv=cv, n_jobs=-1, refit=True
    )
    search.fit(X_train, y_train)
    best = search.best_estimator_
    fitted[name] = best
    y_pred = best.predict(X_test)
    predictions[name] = y_pred
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred) * 100,
        "Precision": precision_score(y_test, y_pred, pos_label=1) * 100,
        "Recall": recall_score(y_test, y_pred, pos_label=1) * 100,
        "F1-Score": f1_score(y_test, y_pred, pos_label=1) * 100,
        "CV_F1_mean": search.best_score_ * 100,
        "CV_F1_std": search.cv_results_["std_test_score"][search.best_index_] * 100,
    })
    print("Best params:", search.best_params_)
    print(f"CV F1: {rows[-1]['CV_F1_mean']:.2f}% ± {rows[-1]['CV_F1_std']:.2f}%")
    print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))

## 6. Performance Evaluation & Comparative Analysis

In [ ]:
results = pd.DataFrame(rows)
results[["Accuracy", "Precision", "Recall", "F1-Score", "CV_F1_mean", "CV_F1_std"]] = results[
    ["Accuracy", "Precision", "Recall", "F1-Score", "CV_F1_mean", "CV_F1_std"]
].round(2)
results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, predictions):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=ax,
        xticklabels=["Benign", "Malignant"],
        yticklabels=["Benign", "Malignant"],
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = results[["Model", "Accuracy", "Precision", "Recall", "F1-Score"]].melt(
    id_vars="Model", var_name="Metric", value_name="Score (%)"
)
plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df, x="Metric", y="Score (%)", hue="Model")
plt.ylim(85, 100)
plt.title("Optimized Model Comparison — Breast Cancer Detection (WBCD)")
plt.legend(title="Model", loc="lower right")
plt.tight_layout()
plt.show()

best = results.loc[results["F1-Score"].idxmax()]
print(
    f"Best model (hold-out F1): {best['Model']} "
    f"(Accuracy={best['Accuracy']:.2f}%, Recall={best['Recall']:.2f}%, F1={best['F1-Score']:.2f}%)"
)

In [ ]:
rf_pipe = fitted["Random Forest"]
selector = rf_pipe.named_steps["select"]
rf = rf_pipe.named_steps["clf"]
selected = [feature_names[i] for i in selector.get_support(indices=True)]
importance = (
    pd.Series(rf.feature_importances_, index=selected)
    .sort_values(ascending=False)
    .head(15)
)
plt.figure(figsize=(8, 5))
importance.plot(kind="barh", color="#2a6f97")
plt.gca().invert_yaxis()
plt.title("Random Forest — Top Feature Importance (tuned)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()